In [1]:
import functions as f
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('data/train_fixed.csv')
X_train = train.drop(columns=['rainfall'])
y_train = train['rainfall']

test = pd.read_csv('data/prev/test_prev.csv')
#wybieramy pierwsze 13 kolumn (czyli bez kombinacji)
X_test = test.iloc[:, :13]

In [3]:
y_train_r = y_train.values.reshape(-1)

y_train_ravel = y_train.to_numpy().ravel()

In [4]:
from sklearn.metrics import roc_auc_score

In [5]:
def evaluate_model(model, X_train, X_valid, y_train, y_valid):
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_valid)[:, 1]
    print(roc_auc_score(y_valid, y_pred))
    return roc_auc_score(y_valid, y_pred)

# CatBoost
private score: 0.89944

In [6]:
from catboost import CatBoostClassifier

In [7]:
categorical_features_indices = [i for i, col in enumerate(X_train.columns) if X_train[col].dtype == 'object']

model_catboost = CatBoostClassifier(
    iterations=600,
    learning_rate=0.05,
    depth=10,
    cat_features=categorical_features_indices,
    verbose=50
)
model_catboost.fit(X_train, y_train_r)
pred_catboost = model_catboost.predict_proba(X_test)[:, 1]

submission_cb = pd.DataFrame({
    'id': 2190 + X_test.index,
    'target': pred_catboost
})

# submission_cb.to_csv('pred/pred_CatBoost1.csv', index=False)

0:	learn: 0.6492822	total: 188ms	remaining: 1m 52s
50:	learn: 0.2337854	total: 2.8s	remaining: 30.1s
100:	learn: 0.1731874	total: 5.54s	remaining: 27.4s
150:	learn: 0.1283424	total: 8.25s	remaining: 24.5s
200:	learn: 0.1001156	total: 10.7s	remaining: 21.3s
250:	learn: 0.0777673	total: 13.6s	remaining: 18.9s
300:	learn: 0.0607964	total: 15.2s	remaining: 15.1s
350:	learn: 0.0484318	total: 16.9s	remaining: 12s
400:	learn: 0.0401092	total: 18.5s	remaining: 9.16s
450:	learn: 0.0338941	total: 21.3s	remaining: 7.02s
500:	learn: 0.0290283	total: 24s	remaining: 4.74s
550:	learn: 0.0250991	total: 26.9s	remaining: 2.4s
599:	learn: 0.0222881	total: 29.2s	remaining: 0us


# AdaBoost
private score: 0.90282

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

In [9]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2),
    n_estimators=600,
    learning_rate=0.05,
    #algorithm='SAMME',
    random_state=42
)

ada_model.fit(X_train, y_train_ravel)
pred_ada = ada_model.predict_proba(X_test)[:, 1]

submission_ada = pd.DataFrame({
    'id': 2190 + X_test.index,
    'target': pred_ada
})
# submission_ada.to_csv('pred/pred_AdaBoost1.csv', index=False)

**2gi model AdaBoost** - private score: 0.89960

In [10]:
ada_model2 = AdaBoostClassifier(n_estimators=400, learning_rate=0.05)
ada_model2.fit(X_train, y_train_ravel)
pred_ada = ada_model2.predict_proba(X_test)[:, 1]

submission_ada2 = pd.DataFrame({
    'id': 2190 + X_test.index,
    'target': pred_ada
})

# submission_ada2.to_csv('pred/pred_AdaBoost2.csv', index=False)